# Lesson 11 Lab — Controllers, Atomics, and the Power/Clock Envelope

**Puzzle:** Why can thousands of parallel updates collapse into a serialized hotspot, and what does that have to do with the rest of the chip?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A complete GPU also needs memory controllers and PHYs, atomic/reduction paths, clock distribution, power delivery, error handling, and global control. Atomics preserve a read-modify-write contract when many threads target shared state. They are indispensable for some algorithms, but concentration on a few addresses creates serialization and fabric/cache pressure even while many execution lanes are available.


## 0. Predict before running

1. Predict which index distribution is slower.
2. Explain why equal update counts can create different contention.
3. Separate the timing evidence from the power-model evidence.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook uses CUDA `scatter_add_` as an atomic-style workload. One candidate spreads updates across many bins; another concentrates them on a small hotspot set. Values, update count, dtype, timing, and reduction result stay fixed. The result shows PyTorch GPU behavior for this operation, not the location or exact design of a dedicated atomic unit. A separate first-order `CV²f` table connects activity to the finite power/clock envelope without claiming telemetry.

- Controllers translate requests into external-memory commands and schedule parallel resources.
- Atomic correctness can impose serialization when addresses collide.
- Clock and power networks constrain all units even though CUDA presents logical concurrency.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["SM updates"] --> B["L1/L2 + NoC"]
  B --> C["atomic read-modify-write"]
  C --> D["memory controller / PHY"]
  E["clock + power delivery"] --> A
  E --> B
  E --> C
```


## 3. Inspect the visual boundary

![Other on-chip structures](../assets/GPU_on_chip_structures_attention_acceleration.png)

- [Four-page printable GPU circuit atlas](../assets/GPU_circuit_structures_from_L2_A4_landscape.pdf)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 11
LESSON_TITLE = 'Controllers, Atomics, and the Power/Clock Envelope'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260824
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | indices distributed across a large output |
| Candidate | indices concentrated on a small set of bins |
| Held constant | update values/count, output size, dtype, warm-up, and event timing |
| Measurements | median latency, collision ratio, checksum, and slowdown |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare dispersed and hotspot CUDA scatter-add updates.


## 6. Inspect the code

Each repeat zeroes the destination before the timed scatter. The two index tensors contain the same number of updates, and checksums verify the same total contribution. Collision ratio is a workload property, not a hardware counter.

Do not run until the code matches the frozen table.


In [2]:
updates = 2**22
bins = 2**18
values = torch.ones(updates, device=DEVICE, dtype=torch.float32)
dispersed_idx = torch.arange(updates, device=DEVICE, dtype=torch.int64) % bins
hotspot_bins = 64
hotspot_idx = torch.arange(updates, device=DEVICE, dtype=torch.int64) % hotspot_bins
destination = torch.zeros(bins, device=DEVICE, dtype=torch.float32)

def run(indices):
    destination.zero_()
    destination.scatter_add_(0, indices, values)

dispersed_samples = cuda_samples(lambda: run(dispersed_idx), repeats=20)
run(dispersed_idx); dispersed_checksum = float(destination.sum().item())
hotspot_samples = cuda_samples(lambda: run(hotspot_idx), repeats=20)
run(hotspot_idx); hotspot_checksum = float(destination.sum().item())
dispersed_median = statistics.median(dispersed_samples)
hotspot_median = statistics.median(hotspot_samples)
metrics = {
    "updates": updates, "bins": bins, "hotspot_bins": hotspot_bins,
    "dispersed_median_ms": dispersed_median, "hotspot_median_ms": hotspot_median,
    "hotspot_slowdown": hotspot_median / dispersed_median,
    "dispersed_collision_ratio": 1 - bins / updates,
    "hotspot_collision_ratio": 1 - hotspot_bins / updates,
    "dispersed_checksum": dispersed_checksum, "hotspot_checksum": hotspot_checksum,
    "dispersed_samples_ms": dispersed_samples, "hotspot_samples_ms": hotspot_samples,
}
analysis = (
    f"Concentrating {updates:,} updates into {hotspot_bins} bins changed median scatter-add "
    f"latency by {metrics['hotspot_slowdown']:.3f}x versus spreading them across {bins:,} bins. "
    "Both routes preserved the total update checksum."
)
print(json.dumps(metrics, indent=2))


{
  "updates": 4194304,
  "bins": 262144,
  "hotspot_bins": 64,
  "dispersed_median_ms": 0.023439999669790268,
  "hotspot_median_ms": 0.2797600030899048,
  "hotspot_slowdown": 11.935153883575458,
  "dispersed_collision_ratio": 0.9375,
  "hotspot_collision_ratio": 0.9999847412109375,
  "dispersed_checksum": 4194304.0,
  "hotspot_checksum": 4194304.0,
  "dispersed_samples_ms": [
    0.03811199963092804,
    0.025119999423623085,
    0.025119999423623085,
    0.020864000543951988,
    0.02364799939095974,
    0.024480000138282776,
    0.02252800017595291,
    0.023584000766277313,
    0.023584000766277313,
    0.023615999147295952,
    0.023455999791622162,
    0.023455999791622162,
    0.021536000072956085,
    0.022655999287962914,
    0.02300800010561943,
    0.02316799946129322,
    0.02300800010561943,
    0.023423999547958374,
    0.023135999217629433,
    0.02300800010561943
  ],
  "hotspot_samples_ms": [
    0.28227201104164124,
    0.28064000606536865,
    0.28060799837112427,
  

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Dispersed median | 0.023 ms |
| Hotspot median | 0.280 ms |
| Hotspot slowdown | 11.935x |
| Dispersed collision ratio | 93.75% |
| Hotspot collision ratio | 100.00% |


## 8. Explain rather than overclaim

Concentrating 4,194,304 updates into 64 bins changed median scatter-add latency by 11.935x versus spreading them across 262,144 bins. Both routes preserved the total update checksum.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 11, "title": 'Controllers, Atomics, and the Power/Clock Envelope', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Reduce address concentration or perform hierarchical local reduction when contention dominates, but preserve the exact update semantics.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 11,
  "title": "Controllers, Atomics, and the Power/Clock Envelope",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260824
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "updates": 4194304,
    "bins": 262144,
    "hotspot_bins": 64,
    "dispersed_median_ms": 0.023439999669790268,
    "hotspot_median_ms": 0.2797600030899048,
    "hotspot_slowdown": 11.935153883575458,
    "dispersed_collision_ratio": 0.9375,
    "hotspot_collision_ratio": 0.9999847412109375,
    "dispersed_checksum": 4194304.0,
    "hotspot_checksum": 4194304.0,
    "dispersed_samples_ms": [
      0.03811199963092804,
      0.025119999423623085,
      0.025119999423623085,
      0.020864000543951988,
      0.02364799939095974,
      0.024480000138282776,
      0.02252800017595291,
      0.023584000766277313,
      0.023584000766277313,
      0.023615999147

## 10. Make the decision

> Reduce address concentration or perform hierarchical local reduction when contention dominates, but preserve the exact update semantics.

**Failure analysis:** `scatter_add_` kernel selection is version-dependent, and caches or internal aggregation may alter scaling. The power model is separate and illustrative.


## 11. Extend the evidence

Sweep bins and update skew, add a two-stage local-reduce candidate, and collect atomic/fabric stall plus board-power telemetry separately.

See [`README.md`](README.md) for the full explanation and references.
